Performance Benchmark

In [ ]:
import requests
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
API_URL = "http://localhost:8000/api/chat"
NPC_ID = "thorin_01"

test_prompts = [
    "Hello.",
    "Who are you and what do you do here?",
    "Can you forge a magical sword for me?",
    "What is your opinion on magic and elves? I heard you hate them.",
    "I need a complete set of steel armor, a shield, and a heavy warhammer. How much will it cost and how long will it take?"
]

ITERATIONS = 5 
results = []

In [ ]:
# Warmup request to ensure the model is loaded and ready
print("Initializing performance tests with a warmup...\n")
requests.post(API_URL, json={"npc_id": NPC_ID, "player_message": "Warmup"})
print("Warmup finished. Beginning performance measurements.\n")

In [ ]:
for i, prompt in enumerate(test_prompts):
    for iteration in range(ITERATIONS):
        print(f"Test {i+1}/{len(test_prompts)} | Iteration {iteration+1}...")
        
        payload = {"npc_id": NPC_ID, "player_message": prompt}
        response = requests.post(API_URL, json=payload).json()
        
        metrics = response.get("metrics", {})
        
        # Converting nanoseconds to seconds and milliseconds
        ns_to_sec = 1e9
        ns_to_ms = 1e6
        
        load_dur_sec = metrics.get("load_duration", 0) / ns_to_sec
        prompt_eval_dur_sec = metrics.get("prompt_eval_duration", 0) / ns_to_sec
        eval_dur_sec = metrics.get("eval_duration", 0) / ns_to_sec
        tokens = metrics.get("eval_count", 0)
        
        # Estimating TTFT (Time To First Token)
        ttft_sec = load_dur_sec + prompt_eval_dur_sec
        
        # TPS (Tokens per second) during the decoding phase
        tps = tokens / eval_dur_sec if eval_dur_sec > 0 else 0
        
        results.append({
            "Prompt_Length_Chars": len(prompt),
            "Tokens_Generated": tokens,
            "TTFT_sec": ttft_sec,
            "Generation_Time_sec": eval_dur_sec,
            "Total_Latency_sec": ttft_sec + eval_dur_sec,
            "TPS": tps
        })

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df_perf = pd.DataFrame(results)
df_perf.to_csv("performance_metrics.csv", index=False)
print("\nTesty zakończone! Zapisano do performance_metrics.csv")

In [ ]:
# ================= Visualization ================= #
sns.set_theme(style="darkgrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: TTFT
sns.boxplot(y="TTFT_sec", data=df_perf, ax=axes[0], color="skyblue")
axes[0].set_title("Time To First Token (TTFT) Estimation")
axes[0].set_ylabel("Seconds")

# Chart 2: Throughput (TPS)
sns.boxplot(y="TPS", data=df_perf, ax=axes[1], color="lightgreen")
axes[1].set_title("Tokens Per Second (TPS)")
axes[1].set_ylabel("Tokens / Second")

plt.tight_layout()
plt.show()